# 03 — Privacy & Governance Demo

**Course:** DEGO 2606 — Data Ecosystems and Governance in Organizations  
**Team:** Group 3 (`dego-2606-group3`)  
**Role:** Governance Officer  
**Dataset:** NovaCred credit applications (502 records, 34 columns)

This notebook demonstrates privacy analysis and governance compliance for NovaCred's ML-driven credit decisioning system. It covers:
- PII identification and classification
- Pseudonymization of sensitive fields
- GDPR compliance assessment
- EU AI Act risk classification
- Concrete governance recommendations

## Agenda

| # | Section | Description |
|---|---|---|
| 1 | [Setup](#1.-Setup) | Imports, logging configuration, reproducibility seed |
| 2 | [Data Loading](#2.-Data-Loading) | Load raw dataset via shared loader |
| 3 | [Pre-processing](#3.-Pre-processing) | Normalise gender, parse dates, deduplicate |
| 4 | [PII Identification](#4.-PII-Identification-&-Classification) | Classify direct identifiers, quasi-identifiers, sensitive attributes |
| 5 | [Pseudonymization Demo](#5.-Pseudonymization-Demo) | SHA-256 hashing of SSN and HMAC of email |
| 6 | [GDPR Mapping](#6.-GDPR-Article-Mapping) | Assess compliance against Art. 5, 6, 17, 22 |
| 7 | [EU AI Act Classification](#7.-EU-AI-Act-Risk-Classification) | Risk tier and obligations |
| 8 | [Governance Controls](#8.-Governance-Controls-&-Recommendations) | Audit trails, oversight, retention policies |
| 9 | [Summary](#9.-Summary) | All findings consolidated |

### 1. Setup
Standard library imports, logging configuration, and reproducibility seed. All subsequent cells depend on these being run first.

In [ ]:
import hashlib
import logging
import os
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

# add project root to path so src/ is importable from notebooks/
sys.path.insert(0, os.path.abspath(".."))

# configure logging to surface info-level messages from the data loader
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(name)s | %(message)s",
)

# reproducibility seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# figure output directory — created if it does not already exist
FIGURES_DIR = "../reports/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Setup complete.")

### 2. Data Loading
Uses the shared `load_raw_data` function from `src/data_loading.py`. No cleaning is applied at load time — all raw issues are preserved intentionally.

In [ ]:
from src.data_loading import load_raw_data

# load raw dataset — path relative to notebooks/ directory
df = load_raw_data("../data/raw/raw_credit_applications.json")

print(f"Loaded dataset: {df.shape[0]} rows \u00d7 {df.shape[1]} columns")
display(df.dtypes.to_frame(name="dtype").T)

### 3. Pre-processing
Resolves known data quality issues that affect privacy analysis: inconsistent gender encoding, mixed date formats, duplicate application IDs, and non-numeric annual income. These mirror the pre-processing steps validated in `02-bias-analysis.ipynb`.

In [ ]:
# normalise inconsistent gender encoding: 'M' -> 'Male', 'F' -> 'Female', blank -> NaN
# raw data contains five distinct representations of two logical categories
gender_map = {"M": "Male", "F": "Female", "": pd.NA}
df["gender_clean"] = df["gender"].replace(gender_map)

print("Gender value counts after normalisation:")
print(df["gender_clean"].value_counts(dropna=False))

In [ ]:
# parse date_of_birth — three mixed formats: YYYY-MM-DD, DD/MM/YYYY, YYYY/MM/DD
# format='mixed' handles all variants; errors='coerce' converts failures to NaT
df["dob_parsed"] = pd.to_datetime(
    df["date_of_birth"], format="mixed", dayfirst=False, errors="coerce"
)
df["age"] = (
    (pd.Timestamp("today") - df["dob_parsed"]).dt.days // 365
).astype("Int64")

print(f"Date parse failures (NaT): {df['dob_parsed'].isna().sum()}")
print(f"Age \u2014 min: {df['age'].min()}, max: {df['age'].max()}, nulls: {df['age'].isna().sum()}")

In [ ]:
# cast annual_income to numeric — 5 records store None as a Python object
df["annual_income_num"] = pd.to_numeric(df["annual_income"], errors="coerce")

# deduplicate by application ID — keep first occurrence
# 2 duplicate ID pairs identified in 01-data-quality.ipynb (app_001, app_042)
n_before = len(df)
df = df.drop_duplicates(subset="id", keep="first").reset_index(drop=True)

print(f"Records removed (duplicate IDs): {n_before - len(df)}")
print(f"Records after deduplication: {len(df)}")

### 4. PII Identification & Classification
Under GDPR, personal data is any information that can identify a natural person directly or indirectly (Article 4(1)). This section classifies each column in the dataset according to three tiers:

- **Direct identifiers** — uniquely identify an individual on their own (e.g. SSN, email)
- **Quasi-identifiers** — individually innocuous but linkable to identify individuals when combined
- **Sensitive / special-category data** — warrants heightened protection under GDPR Article 9 or context-specific regulation
- **Non-PII** — financial and behavioural features with no direct identifying power

In [ ]:
def classify_pii_fields(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Classify each column in the NovaCred dataset by GDPR personal-data tier.

    Parameters
    ----------
    dataframe : pd.DataFrame
        Flat NovaCred credit applications DataFrame returned by load_raw_data.

    Returns
    -------
    pd.DataFrame
        Classification table with columns: field, tier, gdpr_basis, notes.

    Notes
    -----
    Tiers follow GDPR Article 4(1) and ICO guidance on personal data categories.
    """
    # static classification table — each entry is manually reviewed against GDPR
    pii_schema = [
        # Direct Identifiers
        {
            "field": "full_name",
            "tier": "Direct Identifier",
            "gdpr_basis": "Art. 4(1) — personal data",
            "notes": "Uniquely identifies a natural person; must be pseudonymised or removed before any sharing.",
        },
        {
            "field": "email",
            "tier": "Direct Identifier",
            "gdpr_basis": "Art. 4(1) — personal data",
            "notes": "Unique contact address; constitutes personal data under GDPR.",
        },
        {
            "field": "ssn",
            "tier": "Direct Identifier",
            "gdpr_basis": "Art. 4(1) — personal data; national ID number",
            "notes": "Social Security Number — high-sensitivity national identifier. Highest re-identification risk.",
        },
        {
            "field": "ip_address",
            "tier": "Direct Identifier",
            "gdpr_basis": "Art. 4(1) — online identifier (CJEU Breyer, C-582/14)",
            "notes": "Constitutes personal data per CJEU ruling when the data controller can link it to an individual.",
        },
        # Quasi-Identifiers
        {
            "field": "date_of_birth",
            "tier": "Quasi-Identifier",
            "gdpr_basis": "Art. 4(1) — indirect identifier; Art. 9 proxy (age)",
            "notes": "Combined with zip_code and gender, DoB enables re-identification (Sweeney 2000: 87% of US population uniquely identified by DoB+gender+ZIP).",
        },
        {
            "field": "zip_code",
            "tier": "Quasi-Identifier",
            "gdpr_basis": "Art. 4(1) — indirect identifier; geographic proxy",
            "notes": "Five-digit ZIP code narrows location to a neighbourhood. Also acts as a proxy for race/ethnicity in US lending contexts.",
        },
        {
            "field": "gender",
            "tier": "Quasi-Identifier / Protected Attribute",
            "gdpr_basis": "Art. 9(1) — special category (sex); EU AI Act Annex III",
            "notes": "Protected characteristic under EU anti-discrimination law (Directive 2004/113/EC) and a fairness-sensitive attribute for credit scoring.",
        },
        # Sensitive Behavioural Spending
        {
            "field": "spending_adult_entertainment",
            "tier": "Sensitive Behavioural Data",
            "gdpr_basis": "Art. 9 proxy (sexual orientation); data minimisation (Art. 5(1)(c))",
            "notes": "May reveal or be used to infer sexual orientation — a special-category attribute. Collection requires explicit justification.",
        },
        {
            "field": "spending_gambling",
            "tier": "Sensitive Behavioural Data",
            "gdpr_basis": "Art. 5(1)(c) — data minimisation; Art. 5(1)(b) — purpose limitation",
            "notes": "Gambling behaviour is a health/social indicator. Inclusion in a credit model must be justified against the purpose limitation principle.",
        },
        {
            "field": "spending_alcohol",
            "tier": "Sensitive Behavioural Data",
            "gdpr_basis": "Art. 5(1)(c) — data minimisation",
            "notes": "Alcohol spending may infer health status. No clear credit-relevance justification documented.",
        },
        # Non-PII Financial Features
        {
            "field": "annual_income",
            "tier": "Non-PII (Financial)",
            "gdpr_basis": "Art. 6(1)(b) — necessary for contract performance",
            "notes": "Core credit-underwriting variable; lawful basis is contract performance.",
        },
        {
            "field": "credit_history_months",
            "tier": "Non-PII (Financial)",
            "gdpr_basis": "Art. 6(1)(b) — necessary for contract performance",
            "notes": "Standard credit-underwriting metric.",
        },
        {
            "field": "debt_to_income",
            "tier": "Non-PII (Financial)",
            "gdpr_basis": "Art. 6(1)(b) — necessary for contract performance",
            "notes": "Standard credit-underwriting metric.",
        },
        {
            "field": "savings_balance",
            "tier": "Non-PII (Financial)",
            "gdpr_basis": "Art. 6(1)(b) — necessary for contract performance",
            "notes": "Standard credit-underwriting metric.",
        },
        {
            "field": "loan_approved",
            "tier": "Non-PII (Decision Output)",
            "gdpr_basis": "Art. 22 — automated decision; Art. 6(1)(b)",
            "notes": "Automated decision subject to Art. 22 right to explanation and human review.",
        },
        {
            "field": "interest_rate",
            "tier": "Non-PII (Decision Output)",
            "gdpr_basis": "Art. 22 — automated decision",
            "notes": "Pricing output of the automated model; subject to the same Art. 22 obligations as the approval decision.",
        },
    ]

    return pd.DataFrame(pii_schema)


pii_table = classify_pii_fields(df)
display(pii_table)

In [ ]:
# count PII fields by tier
tier_counts = pii_table["tier"].value_counts().rename_axis("tier").reset_index(name="field_count")

print("PII tier distribution:")
display(tier_counts)

In [ ]:
# check population coverage for each direct identifier
# high coverage means re-identification risk applies to nearly all records
direct_id_cols = ["full_name", "email", "ssn", "ip_address"]

coverage = {
    col: {
        "present": df[col].notna().sum(),
        "missing": df[col].isna().sum(),
        "coverage_pct": round(df[col].notna().mean() * 100, 1),
    }
    for col in direct_id_cols
}

coverage_df = pd.DataFrame(coverage).T
print("Direct identifier coverage (n=500 after deduplication):")
display(coverage_df)

In [ ]:
# check population coverage for sensitive spending categories
# even sparse presence requires a documented lawful basis under GDPR
sensitive_cols = ["spending_adult_entertainment", "spending_gambling", "spending_alcohol"]

sensitive_coverage = {
    col: {
        "records_with_data": df[col].notna().sum(),
        "coverage_pct": round(df[col].notna().mean() * 100, 1),
        "mean_amount": round(df[col].mean(), 2),
    }
    for col in sensitive_cols
    if col in df.columns
}

print("Sensitive spending category coverage:")
display(pd.DataFrame(sensitive_coverage).T)

### 5. Pseudonymization Demo
GDPR Recital 26 and Article 25 (data protection by design) encourage pseudonymisation as a technical safeguard. Pseudonymisation replaces direct identifiers with a reversible surrogate token, so that re-identification requires access to a separately held key.

This section demonstrates two techniques:
1. **One-way hashing (SHA-256)** — applied to `ssn`. Deterministic and irreversible without the original value.
2. **Keyed HMAC hashing** — applied to `email`. The HMAC key acts as a secret that must be stored separately; the output is a pseudonym rather than an anonymisation.
3. **IP generalisation** — zeroing the host octet of IPv4 addresses to reduce precision.

> **Limitation:** Hashing alone does not constitute anonymisation under GDPR. True anonymisation requires k-anonymity, differential privacy, or data generalisation.

In [ ]:
def pseudonymise_sha256(value: str) -> str:
    """
    Replace a string value with its SHA-256 hex digest.

    Parameters
    ----------
    value : str
        Raw string value to pseudonymise (e.g. an SSN).

    Returns
    -------
    str
        64-character lowercase hexadecimal SHA-256 digest,
        or pd.NA if the input is None or NaN.

    Notes
    -----
    SHA-256 is deterministic: the same input always produces the same digest,
    allowing internal record linkage while preventing casual disclosure.
    This is pseudonymisation, not anonymisation.
    """
    if pd.isna(value):
        return pd.NA
    # encode to bytes then hash — strip whitespace to avoid digest mismatches
    return hashlib.sha256(str(value).strip().encode("utf-8")).hexdigest()


def pseudonymise_hmac(value: str, secret_key: bytes) -> str:
    """
    Replace a string value with a keyed HMAC-SHA-256 pseudonym.

    Parameters
    ----------
    value : str
        Raw string to pseudonymise (e.g. an email address).
    secret_key : bytes
        Secret key held by the data controller. Must be stored separately
        from the pseudonymised dataset.

    Returns
    -------
    str
        64-character HMAC-SHA-256 hex digest, or pd.NA if the input is null.

    Notes
    -----
    HMAC adds a layer of security over plain SHA-256: without the secret key,
    an adversary cannot verify whether a candidate value matches the pseudonym.
    """
    if pd.isna(value):
        return pd.NA
    # compute HMAC with SHA-256 using the provided secret key
    mac = hashlib.new("sha256", secret_key + str(value).strip().lower().encode("utf-8"))
    return mac.hexdigest()


def generalise_ip(ip: str) -> str:
    """
    Generalise an IPv4 address to its /24 subnet by zeroing the host octet.

    Parameters
    ----------
    ip : str
        Raw IPv4 address string (e.g. '192.168.1.42').

    Returns
    -------
    str
        Generalised address with last octet replaced by '0' (e.g. '192.168.1.0'),
        or pd.NA if the input is null or does not match IPv4 format.
    """
    if pd.isna(ip):
        return pd.NA
    # match four dot-separated decimal octets
    match = re.fullmatch(r"(\d{1,3}\.\d{1,3}\.\d{1,3})\.(\d{1,3})", str(ip).strip())
    if match:
        return match.group(1) + ".0"  # zero out the host octet
    return pd.NA  # return NA for malformed addresses


print("Pseudonymisation functions defined.")

In [ ]:
# work on a copy so the original columns are preserved for comparison
df_pseudonymised = df.copy()

# SSN: SHA-256 hash
df_pseudonymised["ssn_pseudo"] = df_pseudonymised["ssn"].apply(pseudonymise_sha256)

# Email: HMAC-SHA-256 hash
# in production, SECRET_KEY must be stored in a key-management service (e.g. AWS KMS)
# and never hardcoded in source code
SECRET_KEY = b"novacred-demo-key-2024"  # demonstration value only
df_pseudonymised["email_pseudo"] = df_pseudonymised["email"].apply(
    pseudonymise_hmac, secret_key=SECRET_KEY
)

# full_name: replace with application ID as an opaque reference token
# the name is removed entirely; the record is identified only by its system ID
df_pseudonymised["full_name_pseudo"] = df_pseudonymised["id"]

# IP address: generalise to /24 subnet
df_pseudonymised["ip_generalised"] = df_pseudonymised["ip_address"].apply(generalise_ip)

print("Pseudonymisation applied to ssn, email, full_name, and ip_address.")

In [ ]:
# side-by-side comparison of original and pseudonymised values
# show the first 5 non-null SSN records for illustration
comparison_cols = ["id", "full_name", "full_name_pseudo", "ssn", "ssn_pseudo", "email", "email_pseudo"]
sample = (
    df_pseudonymised[df_pseudonymised["ssn"].notna()][comparison_cols]
    .head(5)
    .reset_index(drop=True)
)

# truncate long hashes for display readability
for col in ["ssn_pseudo", "email_pseudo"]:
    sample[col] = sample[col].apply(
        lambda x: str(x)[:16] + "..." if pd.notna(x) else pd.NA
    )

print("Before / after pseudonymisation (first 5 records with non-null SSN):")
display(sample)

# show IP generalisation separately
ip_sample = (
    df_pseudonymised[df_pseudonymised["ip_address"].notna()][["ip_address", "ip_generalised"]]
    .head(5)
    .reset_index(drop=True)
)
print("\nIP address generalisation (host octet zeroed):")
display(ip_sample)

In [ ]:
# verify that the same input always produces the same hash (determinism check)
test_ssn = "123-45-6789"
hash_a = pseudonymise_sha256(test_ssn)
hash_b = pseudonymise_sha256(test_ssn)

print(f"Determinism check — same input produces same output: {hash_a == hash_b}")
print(f"Hash of '{test_ssn}': {hash_a}")

# verify that different inputs produce different hashes (collision resistance in practice)
hash_c = pseudonymise_sha256("123-45-6780")
print(f"Different inputs produce different outputs: {hash_a != hash_c}")

### 6. GDPR Article Mapping
This section assesses NovaCred's compliance posture against the four most relevant GDPR articles for an automated credit decisioning system.

In [ ]:
# GDPR Article 5 — Principles relating to processing
# Key sub-principles: purpose limitation (5.1.b), data minimisation (5.1.c),
# storage limitation (5.1.e), integrity and confidentiality (5.1.f)

print("=" * 70)
print("GDPR Article 5 — Principles relating to processing")
print("=" * 70)

# data minimisation check: columns without documented credit-underwriting justification
unjustified_cols = [
    "notes",                         # 99.6% missing — purpose unclear
    "spending_adult_entertainment",   # no credit-relevance justification
    "spending_gambling",              # no credit-relevance justification
    "spending_alcohol",               # no credit-relevance justification
]

print("\nArt. 5(1)(c) — Data Minimisation: columns without documented credit-underwriting justification")
for col in unjustified_cols:
    if col in df.columns:
        missing_pct = df[col].isna().mean() * 100
        print(f"  {col:<40} (missing: {missing_pct:.1f}%)")

# purpose limitation check: fields with unclear link to credit decisioning
print("\nArt. 5(1)(b) — Purpose Limitation: fields with unclear link to credit decisioning")
purpose_mismatch = ["ssn", "ip_address", "full_name", "email"]
for col in purpose_mismatch:
    present = df[col].notna().sum()
    print(f"  {col:<20} ({present} records with data)")

print("\nVerdict: POTENTIAL VIOLATION — several fields lack a documented credit-underwriting purpose.")

In [ ]:
# GDPR Article 6 — Lawfulness of processing
# The controller must identify a valid legal basis for each processing activity.

print("=" * 70)
print("GDPR Article 6 — Lawfulness of Processing")
print("=" * 70)

lawfulness_assessment = [
    {
        "field_group": "Financial variables (income, DTI, savings, credit_history)",
        "legal_basis": "Art. 6(1)(b) — contract performance",
        "compliant": "YES",
        "rationale": "Core underwriting variables; directly necessary to assess creditworthiness.",
    },
    {
        "field_group": "full_name, email",
        "legal_basis": "Art. 6(1)(b) — contract performance",
        "compliant": "CONDITIONAL",
        "rationale": "Required for applicant identification and communication; retention beyond decision must be justified.",
    },
    {
        "field_group": "SSN",
        "legal_basis": "Art. 6(1)(c) — legal obligation (AML/KYC)",
        "compliant": "CONDITIONAL",
        "rationale": "Potentially justified under AML compliance, but must be documented. 5 records missing SSN — process incomplete.",
    },
    {
        "field_group": "IP address",
        "legal_basis": "Art. 6(1)(f) — legitimate interests",
        "compliant": "UNCERTAIN",
        "rationale": "Fraud prevention is a legitimate interest, but requires a documented balancing test against data subjects' rights.",
    },
    {
        "field_group": "Sensitive spending (gambling, adult_entertainment, alcohol)",
        "legal_basis": "No documented basis identified",
        "compliant": "NO",
        "rationale": "No clear credit-relevance; collecting and using these features without a lawful basis violates Art. 6.",
    },
]

lawfulness_df = pd.DataFrame(lawfulness_assessment)
display(lawfulness_df)

In [ ]:
# GDPR Article 17 — Right to Erasure ('right to be forgotten')
# Data subjects can request erasure when data is no longer necessary,
# consent is withdrawn, or processing is unlawful.

print("=" * 70)
print("GDPR Article 17 — Right to Erasure")
print("=" * 70)


def find_applicant_records(dataframe: pd.DataFrame, ssn_value: str) -> pd.DataFrame:
    """
    Locate all records associated with a given SSN to support an erasure request.

    Parameters
    ----------
    dataframe : pd.DataFrame
        The NovaCred dataset.
    ssn_value : str
        The SSN of the data subject making the erasure request.

    Returns
    -------
    pd.DataFrame
        All rows where ssn matches ssn_value.
    """
    return dataframe[dataframe["ssn"] == ssn_value][
        ["id", "full_name", "ssn", "email", "loan_approved"]
    ]


# use the first non-null SSN in the dataset as a demonstration subject
example_ssn = df["ssn"].dropna().iloc[0]
erasure_hits = find_applicant_records(df, example_ssn)

print(f"\nErasure request lookup for SSN '{example_ssn}':")
display(erasure_hits)
print(f"Records found: {len(erasure_hits)} — all must be erased or anonymised within 30 days (Art. 17(1)).")

# duplicate ID records are an erasure risk — a deletion by ID may miss the duplicate
dup_ids = df[df["id"].duplicated(keep=False)]["id"].unique()
print(f"\nDuplicate application IDs detected: {list(dup_ids)}")
print("Risk: an erasure workflow that deletes by ID may miss duplicate records if de-duplication was not applied first.")

In [ ]:
# GDPR Article 22 — Automated Individual Decision-Making
# Art. 22 gives data subjects the right NOT to be subject to a decision based
# solely on automated processing that produces legal or similarly significant effects.
# Credit approval/rejection is explicitly cited in Recital 71 as such a decision.

print("=" * 70)
print("GDPR Article 22 — Automated Decision-Making")
print("=" * 70)

# quantify the scope of automated decisions in the dataset
total_decisions = df["loan_approved"].notna().sum()
approvals = int(df["loan_approved"].sum())
rejections = total_decisions - approvals

print(f"\nTotal automated decisions: {total_decisions}")
print(f"  Approvals:   {approvals}  ({approvals/total_decisions:.1%})")
print(f"  Rejections:  {rejections}  ({rejections/total_decisions:.1%})")

# Art. 22(3) requires meaningful information about the logic, significance,
# and consequences of the automated decision
has_reason = int(df.loc[df["loan_approved"] == False, "rejection_reason"].notna().sum())
no_reason  = int(df.loc[df["loan_approved"] == False, "rejection_reason"].isna().sum())

print(f"\nRejected applications WITH documented rejection_reason:    {has_reason} / {rejections} ({has_reason/rejections:.1%})")
print(f"Rejected applications WITHOUT documented rejection_reason: {no_reason} / {rejections} ({no_reason/rejections:.1%})")

print("\nRejection reason distribution:")
display(df.loc[df["loan_approved"] == False, "rejection_reason"].value_counts(dropna=False))

print(f"\nArt. 22 verdict: {no_reason} rejections have no documented reason — GDPR Art. 22(3) violation risk.")

In [ ]:
# visualise GDPR compliance status across the four key articles
gdpr_compliance = {
    "Art. 5 — Minimisation / Purpose": "NON-COMPLIANT",
    "Art. 6 — Lawful Basis": "PARTIAL",
    "Art. 17 — Erasure Readiness": "PARTIAL",
    "Art. 22 — Auto-Decision Explanation": "NON-COMPLIANT",
}

status_colors = {
    "COMPLIANT": "#55A868",
    "PARTIAL": "#FFC107",
    "NON-COMPLIANT": "#DD8452",
}

articles = list(gdpr_compliance.keys())
statuses = list(gdpr_compliance.values())
bar_colors = [status_colors[s] for s in statuses]

fig, ax = plt.subplots(figsize=(9, 3))

bars = ax.barh(articles, [1] * len(articles), color=bar_colors, edgecolor="white", height=0.5)

for bar, status in zip(bars, statuses):
    ax.text(
        0.5, bar.get_y() + bar.get_height() / 2,
        status, ha="center", va="center", fontsize=11, fontweight="bold", color="white",
    )

ax.set_xlim(0, 1)
ax.set_xticks([])
ax.set_title("GDPR Compliance Assessment — NovaCred Credit Decisioning System", fontweight="bold")
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/03_gdpr_compliance_overview.png", dpi=150)
plt.show()

### 7. EU AI Act Risk Classification
The EU AI Act (Regulation 2024/1689) establishes a risk-based framework for AI systems. The risk tier determines the compliance obligations imposed on the provider and deployer.

In [ ]:
print("=" * 70)
print("EU AI Act — Risk Tier Classification")
print("=" * 70)

print("""
System under assessment
-----------------------
Provider:    NovaCred (fintech startup)
System:      ML-driven credit decisioning model
Function:    Automated loan approval/rejection and interest rate assignment
Deployment:  Consumer credit applications in the EU

Classification: HIGH-RISK
Legal basis: Annex III, Point 5(b) of the EU AI Act

  \"AI systems intended to be used to evaluate the creditworthiness of natural
   persons or establish their credit score, with the exception of AI systems
   used for the purpose of detecting financial fraud.\"

Supporting indicators:
  1. Automated approval/rejection has legally significant effects on applicants
     (access to credit, financial inclusion).
  2. System processes protected attributes (gender, age) directly or via proxy
     variables (zip_code), raising Art. 10(5) AI Act concerns.
  3. Bias analysis confirms Disparate Impact ratio = 0.77 (below the 0.80
     four-fifths threshold), evidencing discriminatory outcomes by gender.
""")

In [ ]:
# High-risk obligations under EU AI Act Chapter III, Section 2 (Articles 9-15)
obligations = [
    {
        "article": "Art. 9",
        "obligation": "Risk Management System",
        "description": "Establish and maintain a risk management system throughout the AI system lifecycle.",
        "current_status": "NOT EVIDENCED",
        "evidence_gap": "No documented risk management process found in repository.",
    },
    {
        "article": "Art. 10",
        "obligation": "Data Governance",
        "description": "Training and test data must be relevant, representative, and free of errors. Bias must be identified and addressed.",
        "current_status": "PARTIAL",
        "evidence_gap": "Bias identified (DI = 0.77, DPD = 0.15) but remediation not implemented.",
    },
    {
        "article": "Art. 11",
        "obligation": "Technical Documentation",
        "description": "Maintain technical documentation before market deployment, updated continuously.",
        "current_status": "NOT EVIDENCED",
        "evidence_gap": "No model card, data sheet, or technical specification found.",
    },
    {
        "article": "Art. 12",
        "obligation": "Record-Keeping / Audit Logs",
        "description": "Automatic logging of events throughout the system lifecycle to enable post-hoc auditing.",
        "current_status": "NOT EVIDENCED",
        "evidence_gap": "processing_timestamp is 87.6% missing — no reliable audit trail.",
    },
    {
        "article": "Art. 13",
        "obligation": "Transparency",
        "description": "AI system must be sufficiently transparent. Deployers must be given instructions for use.",
        "current_status": "NOT EVIDENCED",
        "evidence_gap": "No transparency documentation provided to credit officers.",
    },
    {
        "article": "Art. 14",
        "obligation": "Human Oversight",
        "description": "Effective human oversight measures must be in place. Humans must be able to intervene and override decisions.",
        "current_status": "NOT EVIDENCED",
        "evidence_gap": f"No override mechanism documented; {no_reason} rejections have no recorded rejection_reason.",
    },
    {
        "article": "Art. 15",
        "obligation": "Accuracy, Robustness & Cybersecurity",
        "description": "High-risk AI systems must achieve appropriate accuracy levels and be robust against errors and adversarial inputs.",
        "current_status": "NOT EVIDENCED",
        "evidence_gap": "No model performance metrics, accuracy benchmarks, or adversarial testing documented.",
    },
]

obligations_df = pd.DataFrame(obligations)
print("EU AI Act — High-Risk System Obligations Assessment:")
display(obligations_df)

In [ ]:
# visualise the compliance gap across AI Act obligations
status_colors_ai = {"NOT EVIDENCED": "#DD8452", "PARTIAL": "#FFC107", "COMPLIANT": "#55A868"}

fig, ax = plt.subplots(figsize=(9, 4))

labels = obligations_df["article"] + " — " + obligations_df["obligation"]
bars = ax.barh(
    labels,
    [1] * len(obligations_df),
    color=[status_colors_ai[s] for s in obligations_df["current_status"]],
    edgecolor="white",
    height=0.55,
)

for bar, status in zip(bars, obligations_df["current_status"]):
    ax.text(
        0.5, bar.get_y() + bar.get_height() / 2,
        status, ha="center", va="center", fontsize=9, fontweight="bold", color="white",
    )

ax.set_xlim(0, 1)
ax.set_xticks([])
ax.set_title("EU AI Act Compliance Gap — NovaCred (High-Risk System)", fontweight="bold")
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/03_ai_act_compliance_gap.png", dpi=150)
plt.show()

### 8. Governance Controls & Recommendations
Based on the GDPR and EU AI Act assessments above, this section proposes concrete, actionable governance controls mapped to the identified compliance gaps. Controls are prioritised by urgency: **Critical** (legal violation risk), **High** (significant risk), and **Medium** (best practice).

In [ ]:
recommendations = [
    # Critical
    {
        "priority": "Critical",
        "area": "Data Minimisation",
        "issue": "Sensitive spending categories (gambling, adult_entertainment, alcohol) collected without documented lawful basis.",
        "recommendation": "Immediately remove spending_gambling, spending_adult_entertainment, and spending_alcohol from the model feature set. Commission a Data Protection Impact Assessment (DPIA) before re-introducing any behavioural spending data.",
        "regulatory_ref": "GDPR Art. 5(1)(b)(c); Art. 35",
    },
    {
        "priority": "Critical",
        "area": "Bias Remediation",
        "issue": "Gender Disparate Impact ratio = 0.77 (below 0.80 four-fifths threshold); Demographic Parity Difference = 0.15.",
        "recommendation": "Suspend model deployment for new applications until bias is remediated. Apply pre-processing (re-sampling) or in-processing (fairness constraints) techniques. Re-evaluate DI ratio after remediation before re-deployment.",
        "regulatory_ref": "EU AI Act Art. 10; GDPR Art. 22; Directive 2004/113/EC",
    },
    {
        "priority": "Critical",
        "area": "Decision Explanation",
        "issue": f"{no_reason} / {rejections} rejections ({no_reason/rejections:.0%}) have no documented rejection_reason — violates Art. 22(3).",
        "recommendation": "Implement mandatory rejection_reason logging for every automated rejection. Provide applicants with a human-readable explanation within 30 days of the decision.",
        "regulatory_ref": "GDPR Art. 22(3); EU AI Act Art. 13",
    },
    # High
    {
        "priority": "High",
        "area": "Pseudonymisation",
        "issue": "SSN, full_name, email, and ip_address stored in plaintext alongside decision outputs.",
        "recommendation": "Apply pseudonymisation (SHA-256 for SSN, HMAC for email) at ingestion. Store the pseudonymisation key in a dedicated key-management service (e.g. AWS KMS). Analytical datasets must never contain direct identifiers.",
        "regulatory_ref": "GDPR Art. 5(1)(f); Art. 25; Recital 26",
    },
    {
        "priority": "High",
        "area": "Audit Trail",
        "issue": "processing_timestamp missing for 87.6% of records — no reliable audit trail.",
        "recommendation": "Enforce non-nullable processing_timestamp at database ingestion level. Append system-generated timestamps (UTC) for every record creation, update, and decision event. Retain logs for minimum 5 years per AML obligations.",
        "regulatory_ref": "EU AI Act Art. 12; GDPR Art. 5(2)",
    },
    {
        "priority": "High",
        "area": "Human Oversight",
        "issue": "No documented human review or override mechanism for automated credit decisions.",
        "recommendation": "Implement a human review queue for: (a) all rejections, (b) borderline approvals (model confidence < 70%), and (c) cases flagged by the fairness monitoring system. Designate a named credit review officer responsible for final decisions.",
        "regulatory_ref": "EU AI Act Art. 14; GDPR Art. 22(2)(b)",
    },
    {
        "priority": "High",
        "area": "ZIP Code Proxy",
        "issue": "ZIP code is strongly correlated with gender (chi-square p < 0.0001) — acts as a geographic proxy for a protected attribute.",
        "recommendation": "Remove zip_code from the model feature set immediately. If geographic risk is relevant, use only aggregated regional data (e.g. CBSA-level unemployment rate) that cannot serve as an individual-level proxy.",
        "regulatory_ref": "EU AI Act Art. 10(2)(f); GDPR Art. 22",
    },
    # Medium
    {
        "priority": "Medium",
        "area": "Erasure Workflow",
        "issue": "Duplicate application IDs (app_001, app_042) create a risk that erasure requests delete only one of the duplicate records.",
        "recommendation": "Implement deduplication as a mandatory pre-processing step. Erasure requests should search by SSN hash (not application ID) to ensure all records for a data subject are located and removed within the 30-day deadline.",
        "regulatory_ref": "GDPR Art. 17",
    },
    {
        "priority": "Medium",
        "area": "Data Retention Policy",
        "issue": "No retention schedule documented for credit application records.",
        "recommendation": "Define retention periods: (a) approved applications: 7 years for AML compliance; (b) rejected applications: 2 years for dispute resolution; (c) direct identifiers: pseudonymise or delete after 90 days post-decision.",
        "regulatory_ref": "GDPR Art. 5(1)(e); 4AMLD Art. 40",
    },
    {
        "priority": "Medium",
        "area": "Model Documentation",
        "issue": "No model card or technical documentation found — required for EU AI Act conformity.",
        "recommendation": "Produce a model card covering: (a) intended use, (b) training data sources and preprocessing, (c) performance metrics by demographic group, (d) known limitations and failure modes, (e) bias mitigation measures applied.",
        "regulatory_ref": "EU AI Act Art. 11; Art. 13",
    },
]

recommendations_df = pd.DataFrame(recommendations)
print("Governance Controls & Recommendations:")
display(recommendations_df[["priority", "area", "recommendation", "regulatory_ref"]])

In [ ]:
# bar chart: count of recommendations by priority tier
priority_counts = recommendations_df["priority"].value_counts().reindex(["Critical", "High", "Medium"])

priority_colors = {"Critical": "#DD8452", "High": "#FFC107", "Medium": "#4C72B0"}

fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.bar(
    priority_counts.index,
    priority_counts.values,
    color=[priority_colors[p] for p in priority_counts.index],
    edgecolor="white",
    width=0.5,
)

for bar, count in zip(bars, priority_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
        str(count), ha="center", va="bottom", fontsize=12, fontweight="bold",
    )

ax.set_title("Governance Recommendations by Priority", fontweight="bold")
ax.set_xlabel("Priority Tier")
ax.set_ylabel("Number of Recommendations")
ax.set_ylim(0, priority_counts.max() + 1)

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/03_governance_recommendations_by_priority.png", dpi=150)
plt.show()

### 9. Summary
All privacy, GDPR, and governance findings consolidated into a single reference table.

In [ ]:
summary_findings = pd.DataFrame({
    "Finding": [
        "Direct identifiers present in dataset",
        "Quasi-identifiers present in dataset",
        "Sensitive spending categories collected",
        "SSN population coverage",
        "IP address population coverage",
        "GDPR Art. 5 — data minimisation",
        "GDPR Art. 6 — lawful basis",
        "GDPR Art. 17 — erasure readiness",
        "GDPR Art. 22 — rejection reason coverage",
        "EU AI Act risk tier",
        "EU AI Act obligations met (of 7)",
        "Duplicate ID erasure risk",
        "Retention policy documented",
        "Human oversight mechanism evidenced",
    ],
    "Value": [
        "4 (full_name, email, ssn, ip_address)",
        "3 (date_of_birth, zip_code, gender)",
        "3 (gambling, adult_entertainment, alcohol)",
        f"{df['ssn'].notna().mean():.1%} of records",
        f"{df['ip_address'].notna().mean():.1%} of records",
        "NON-COMPLIANT",
        "PARTIAL",
        "PARTIAL",
        f"{has_reason}/{rejections} ({has_reason/rejections:.0%})",
        "HIGH-RISK (Annex III \u00a75(b))",
        "1 / 7 (partial)",
        "YES — 2 duplicate ID pairs",
        "NO",
        "NOT EVIDENCED",
    ],
    "Status": [
        "Action Required",
        "Action Required",
        "Critical",
        "Pseudonymise at Ingestion",
        "Pseudonymise / Generalise",
        "Critical",
        "Partial",
        "Partial",
        "Critical",
        "High-Risk Obligations Apply",
        "Significant Gap",
        "High",
        "Medium",
        "High",
    ],
})

display(summary_findings)

### Conclusions

The NovaCred credit decisioning dataset presents **significant privacy and governance risks** across all three regulatory dimensions assessed in this notebook.

**Privacy (GDPR)**
- Four direct identifiers (SSN, email, full_name, IP address) are stored in plaintext alongside model outputs, violating the data protection by design principle (Art. 25).
- Three sensitive spending categories (gambling, adult entertainment, alcohol) are collected without a documented lawful basis, breaching Art. 5(1)(b)(c).
- 87.6% of records have no `processing_timestamp`, making it impossible to demonstrate accountability (Art. 5(2)) or enforce retention deadlines.
- Over 40% of automated rejections lack a `rejection_reason`, directly violating the right to explanation under Art. 22(3).

**EU AI Act**
- NovaCred's credit decisioning system is a **high-risk AI system** under Annex III, §5(b). It must comply with Articles 9–15 before deployment or continued operation.
- As of audit, 6 of 7 high-risk obligations show no evidence of implementation. The single partial compliance (Art. 10 data governance) still requires bias remediation.

**Governance Controls**
- **3 Critical**, **4 High**, and **3 Medium** priority recommendations are raised.
- The most urgent actions are: (1) remove sensitive spending features from the model, (2) suspend the biased model pending remediation (DI = 0.77), and (3) implement mandatory rejection_reason logging.

NovaCred should treat this audit as the basis for an immediate **corrective action plan** with named owners and deadlines for each recommendation.